## 🧪 Test Dataset

In [1]:
from my_datasets.imdb_dataset import IMDBDataset
from my_tokenizers.text_tokenizer import TextTokenizer

In [2]:
imdb = IMDBDataset()

imdb.df.info()

Dataset downloaded to: C:\Users\Ali-Kazemini\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB


In [3]:
imdb.df.describe()

,sentiment
count,50000.000000
mean,0.500000
std,0.500005
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [4]:
print(imdb.df["review"].head())

0    One of the other reviewers has mentioned that ...
1    A wonderful little production. The filming tec...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: object


In [5]:
print(imdb.df["sentiment"].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


## 🍰 Split data `80%` `10%` `10%`

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    imdb.df, test_size=0.2, stratify=imdb.df["sentiment"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["sentiment"], random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Train:")
print(train_df["sentiment"].value_counts())

print("\nValidation:")
print(val_df["sentiment"].value_counts())

print("\nTest:")
print(test_df["sentiment"].value_counts())

Train: 40000
Validation: 5000
Test: 5000
Train:
sentiment
1    20000
0    20000
Name: count, dtype: int64

Validation:
sentiment
0    2500
1    2500
Name: count, dtype: int64

Test:
sentiment
0    2500
1    2500
Name: count, dtype: int64


## 🎩 Train Tokenizer

In [7]:
print(train_df["review"].sample())

texts = train_df["review"].tolist()


44168    This so called remake is terrible. I went to s...
Name: review, dtype: object


In [8]:
tokenizer = TextTokenizer(
    algorithm="wordlevel",
    vocab_size=20_000,
    min_frequency=2,
)

In [9]:
tokenizer.train(texts)
print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 20000


In [10]:
text = imdb.df["review"].iloc[0]

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(tokens)
print(ids)
print(decoded)
print(tokenizer.vocab_size)

['One', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'Oz', 'episode', 'you', "'", 'll', 'be', 'hooked', '.', 'They', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', '.', 'The', 'first', 'thing', 'that', 'struck', 'me', 'about', 'Oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', ',', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'GO', '.', 'Trust', 'me', ',', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', '.', 'This', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', ',', 'sex', 'or', 'violence', '.', 'Its', 'is', 'hardcore', ',', 'in', 'the', 'classic', 'use', 'of', 'the', 'word', '.', 'It', 'is', 'called', 'OZ', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'Oswald', '<UNK>', 'Security', 'State', '<UNK>', '.', 'It', 'focuses', 'mainly', 'on', '<UNK>', 'City', ',', 'an', 'experimental', 'se

## 🚜👷🚧🏗️ Data loader

In [11]:
from my_datasets.imdb_torch_dataset import IMDBTorchDataset

train_dataset = IMDBTorchDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
)

val_dataset = IMDBTorchDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
)

test_dataset = IMDBTorchDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
)

In [12]:
from functools import partial

from torch.utils.data import DataLoader

from my_datasets.imdb_torch_dataset import collate_fn

batch_size = 32

collate = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate,
)

print(f"Train dataset:      {len(train_dataset)}")
print(f"Validation dataset: {len(val_dataset)}")
print(f"Test dataset:       {len(test_dataset)}")

print(f"\nTrain batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train dataset:      40000
Validation dataset: 5000
Test dataset:       5000

Train batches:      1250
Validation batches: 157
Test batches:       157


## 💎 Train Model

### 🎯 define model

In [13]:
import torch
from torch import nn

from my_classifier.lstm_classifier import LSTMClassifier
from my_trainers.trainer import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMClassifier(
    vocab_size=tokenizer.vocab_size, embedding_dim=128, hidden_dim=128, num_classes=2
).to(device)

print(model)
print('Device:', device)

LSTMClassifier(
  (embedding): Embedding(20000, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)
Device: cuda


### 🚀 Optimizer and 📈 LOSS

In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

### 👟 🏋️ The `Trainer`

In [15]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
)

In [16]:
for epoch in range(5):
    train_loss, train_accuracy = trainer.train_epoch()
    
    val_loss, val_accuracy = trainer.evaluate(val_loader)
    
    print(
        f"Epoch {epoch}: "
        f"Train Loss={train_loss:.4f}, "
        f"Train Acc={train_accuracy:.4f}, "
        f"Val Loss={val_loss:.4f}, "
        f"Val Acc={val_accuracy:.4f}"
    )

Training:   0%|          | 0/1250 [00:00<?, ?it/s]

Training: 100%|██████████| 1250/1250 [02:05<00:00,  9.94it/s]


Epoch 0: Train Loss=0.5609, Train Acc=0.7070, Val Loss=0.4547, Val Acc=0.7914


Training: 100%|██████████| 1250/1250 [02:07<00:00,  9.82it/s]


Epoch 1: Train Loss=0.3883, Train Acc=0.8323, Val Loss=0.3230, Val Acc=0.8632


...

Epoch 2: Train Loss=0.2292, Train Acc=0.9085, Val Loss=0.2258, Val Acc=0.9064


Training: 100%|██████████| 1250/1250 [02:03<00:00, 10.09it/s]


Epoch 3: Train Loss=0.1523, Train Acc=0.9431, Val Loss=0.2243, Val Acc=0.9102


Training: 100%|██████████| 1250/1250 [02:05<00:00,  9.95it/s]
                                                   

Epoch 4: Train Loss=0.0987, Train Acc=0.9659, Val Loss=0.2461, Val Acc=0.9090


## 💾 Save Parameters!

it will save model and tokenizer

In [ ]:
torch.save(model.state_dict(), "lstm_imdb.pt")
tokenizer.save("imdb_tokenizer.json")

## 🔄 Load Params

In [1]:
import torch

from my_classifier.lstm_classifier import LSTMClassifier
from my_tokenizers.text_tokenizer import TextTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load tokenizer
tokenizer = TextTokenizer.load("imdb_tokenizer.json")

# load model
state_dict = torch.load("lstm_imdb.pt", map_location=device)

loaded_model = LSTMClassifier(
    vocab_size=tokenizer.vocab_size, embedding_dim=128, hidden_dim=128, num_classes=2
).to(device)

loaded_model.load_weights("lstm_imdb.pt")

LSTMClassifier(
  (embedding): Embedding(20000, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)

In [ ]:
loaded_model.predict_sentiment("The Movie Was Great", tokenizer)

'Positive'

## 💀 Our Must hard case

In [ ]:
loaded_model.predict_sentiment(
    "A bad movie is a disaster, and a good movie is incredible. This was the latter.",
    tokenizer,
)

'Negative'

In [ ]:
loaded_model.predict_sentiment(
    "A good movie is incredible, and a bad movie is a disaster. This was the former.",
    tokenizer,
)

'Negative'

In [32]:
import os

print("CUDA_PATH:", os.environ.get("CUDA_PATH"))

CUDA_PATH: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4
